# Demo inferensi segmentasi instance

Notebook ini memuat bobot terbaik M1 (YOLOv8n-seg), M2 (YOLO11n-seg), dan M3 (Mask R-CNN R50-FPN) dari folder `hasil_cpmk3`, lalu menjalankan ketiganya pada satu foto baru. Ambang skor dan ukuran input bisa diubah langsung di sel demo.

Cara menyiapkan bobot, pilih salah satu:

- **Kaggle**: Add Input → Your Work → output notebook `segmentasi_instance_indoor`, lalu Run All. GPU tidak wajib.
- **Colab**: jalankan sel 2, lalu unggah `hasil_cpmk3.zip` saat diminta.
- **Laptop**: taruh `hasil_cpmk3.zip` di folder yang sama dengan notebook ini.

Foto baru diunggah di sel 3. Di Kaggle, tambahkan foto lewat Add Input → Upload sebagai dataset kecil, atau isi `FOTO` dengan path-nya.

In [ ]:
import importlib, subprocess, sys
for pkg, mod in [("ultralytics", "ultralytics"), ("pillow-heif", "pillow_heif"), ("opencv-python-headless", "cv2")]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import os, glob, json, zipfile, time
from pathlib import Path
import numpy as np, pandas as pd, cv2, torch, matplotlib.pyplot as plt
import torchvision.transforms.functional as TF
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener; register_heif_opener()
from ultralytics import YOLO
from torchvision.models.detection import maskrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor

DEV = "cuda" if torch.cuda.is_available() else "cpu"
IN_COLAB = "google.colab" in sys.modules
print("Perangkat:", DEV, "| Colab" if IN_COLAB else "")

In [ ]:
# Cari folder hasil_cpmk3: input Kaggle, folder lokal, atau ekstrak dari hasil_cpmk3.zip
def cari_hasil():
    # Folder hasil dikenali dari isinya (kelas_bersih.json + runs/), karena dataset Kaggle bisa diekstrak tanpa folder induk
    for root in ["/kaggle/input", "."]:
        for kb in sorted(glob.glob(f"{root}/**/kelas_bersih.json", recursive=True)):
            if Path(kb).parent.joinpath("runs").exists(): return Path(kb).parent
    zp = next(iter(glob.glob("/kaggle/input/**/hasil_cpmk3.zip", recursive=True) + glob.glob("**/hasil_cpmk3.zip", recursive=True)), None)
    if zp is None and IN_COLAB:
        from google.colab import files
        print("Unggah hasil_cpmk3.zip"); zp = next(iter(files.upload()))
    if zp:
        with zipfile.ZipFile(zp) as z: z.extractall(".")
        return Path("hasil_cpmk3")
    raise FileNotFoundError("Folder hasil_cpmk3 tidak ditemukan. Lihat petunjuk di sel pertama.")

HASIL = cari_hasil()
NAMES = json.loads((HASIL/"kelas_bersih.json").read_text())["names"]; NC = len(NAMES)
AP = pd.read_csv(HASIL/"ap_per_kelas.csv")
print("Folder hasil:", HASIL, "|", NC, "kelas")

In [ ]:
def build_mrcnn(imgsz):
    m = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None, min_size=imgsz, max_size=imgsz, box_detections_per_img=100)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NC + 1)
    m.roi_heads.mask_predictor = MaskRCNNPredictor(m.roi_heads.mask_predictor.conv5_mask.in_channels, 256, NC + 1)
    return m

class YoloSeg:
    def __init__(self, w): self.m = YOLO(str(w))
    def __call__(self, img, conf, imgsz):
        r = self.m.predict(img, imgsz=imgsz, conf=conf, iou=0.7, max_det=100, retina_masks=True, verbose=False,
                           device=0 if DEV == "cuda" else "cpu")[0]
        if r.masks is None or len(r.boxes) == 0: return np.zeros((0, *img.shape[:2]), bool), np.zeros(0, int), np.zeros(0)
        return r.masks.data.bool().cpu().numpy(), r.boxes.cls.long().cpu().numpy(), r.boxes.conf.float().cpu().numpy()

class MaskRCNNSeg:
    def __init__(self, w):
        self.m = build_mrcnn(640); self.m.load_state_dict(torch.load(w, map_location="cpu")); self.m.to(DEV).eval()
    @torch.no_grad()
    def __call__(self, img, conf, imgsz):
        self.m.transform.min_size, self.m.transform.max_size = (imgsz,), imgsz
        self.m.roi_heads.score_thresh = conf
        o = self.m([TF.to_tensor(img[..., ::-1].copy()).to(DEV)])[0]
        return (o["masks"][:, 0] > 0.5).cpu().numpy(), (o["labels"] - 1).cpu().numpy(), o["scores"].cpu().numpy()

RUNS = HASIL/"runs"
MODELS = {"M1 YOLOv8n-seg": YoloSeg(RUNS/"M1_yolov8n-seg_640/weights/best.pt"),
          "M2 YOLO11n-seg": YoloSeg(RUNS/"M2_yolo11n-seg_640/weights/best.pt"),
          "M3 Mask R-CNN": MaskRCNNSeg(RUNS/"M3_maskrcnn-r50fpn_640/weights/best.pt")}
print("Model siap:", list(MODELS))

In [ ]:
# Foto baru. Kosongkan FOTO untuk memilih otomatis foto terbaru di /kaggle/input atau folder ini, atau unggah di Colab
FOTO = ""

def muat_foto(path):
    im = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    im.thumbnail((1280, 1280))   # foto HP diperkecil agar cepat, sama seperti evaluasi test set sendiri
    return np.array(im)[..., ::-1].copy()

def foto_terbaru():
    # Foto yang paling baru diunggah di /kaggle/input atau folder ini (gambar hasil evaluasi diabaikan)
    if IN_COLAB:
        from google.colab import files
        return next(iter(files.upload()))
    cand = [p for root in ["/kaggle/input", "."] for ext in ("jpg", "jpeg", "png", "heic", "JPG", "JPEG", "HEIC")
            for p in glob.glob(f"{root}/**/*.{ext}", recursive=True) if "/gambar/" not in p.replace(os.sep, "/") and "runs" not in p]
    return max(cand, key=os.path.getmtime) if cand else ""

FOTO = FOTO or foto_terbaru()
print("Foto:", FOTO)
IMG = muat_foto(FOTO)

In [ ]:
def warna(c):
    r, g, b = plt.matplotlib.colors.hsv_to_rgb([(c * 0.618034) % 1.0, 0.75, 0.95]); return (int(b * 255), int(g * 255), int(r * 255))

def gambar(img, masks, labels, scores):
    out = img.copy()
    for m, c, s in zip(masks, labels, scores):
        col = np.array(warna(int(c)), np.uint8); out[m] = (0.45 * col + 0.55 * out[m]).astype(np.uint8)
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, cs, -1, tuple(int(v) for v in col), 2)
        ys, xs = np.nonzero(m)
        if len(xs):
            org, txt = (int(xs.min()), max(14, int(ys.min()) + 14)), f"{NAMES[int(c)]} {s:.2f}"
            cv2.putText(out, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 3, cv2.LINE_AA)
            cv2.putText(out, txt, org, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (20, 20, 20), 1, cv2.LINE_AA)
    return out[..., ::-1]

def demo(img=None, conf=0.25, imgsz=640):
    img = IMG if img is None else img
    fig, axs = plt.subplots(1, 3, figsize=(20, 7)); rows = []
    for ax, (nama, model) in zip(axs, MODELS.items()):
        t0 = time.perf_counter(); masks, labels, scores = model(img, conf, imgsz); ms = (time.perf_counter() - t0) * 1000
        ax.imshow(gambar(img, masks, labels, scores)); ax.axis("off")
        ax.set_title(f"{nama}: {len(labels)} objek | {ms:.0f} ms")
        for c in labels: rows.append({"model": nama, "kelas": NAMES[int(c)]})
    fig.suptitle(f"ambang skor {conf} | input {imgsz}"); plt.tight_layout(); plt.show()
    if rows: display(pd.DataFrame(rows).value_counts().unstack(0, fill_value=0))

demo(conf=0.25, imgsz=640)

Contoh perubahan yang bisa diminta saat sesi:

- Ambang skor dinaikkan atau diturunkan: `demo(conf=0.5)` atau `demo(conf=0.1)`
- Ukuran input sesuai variabel NRP: `demo(imgsz=320)` atau `demo(imgsz=960)`
- AP satu kelas dari tabel hasil: `ap_kelas("chair")`

In [ ]:
def ap_kelas(kelas, imgsz=640):
    d = AP[(AP.kelas == kelas) & (AP.imgsz == imgsz)]
    if d.empty: print(f"Kelas '{kelas}' tidak ada. Pilihan:", sorted(AP.kelas.unique())); return
    t = d.pivot_table(index="model", columns="test_set", values="AP_mask").round(4)
    display(t)

ap_kelas("chair")

## ▶ FOTO BARU SAAT SESI

1. Panel kanan: **Add Input → Upload** → masukkan foto → **Create**
2. Klik sel di bawah ini, lalu tekan **Shift+Enter**. Sel ini memakai foto yang paling baru diunggah.

In [ ]:
FOTO = foto_terbaru(); print("Foto:", FOTO)
IMG = muat_foto(FOTO)
demo()